# CS-317: Reinforcement Learning — Lab 8
## Model-Free Prediction with Temporal Difference Learning
**BS AI-1**

This notebook provides the corridor environment of Lab 7, the two policies, the exact reference values and helper functions. Complete every `# TODO` cell, run all cells top to bottom without errors, and answer the written questions in the markdown cells provided for them.

- Set your personal **SEED** (the last three digits of your roll number) in the first code cell below. Every random experiment must use it, and your written answers must refer to your own numbers.
- **Predict first** cells must be filled in *before* you run the code beneath them.
- The exact values `V_exact_U` and `V_exact_R` are for **checking only**. No estimate may be computed from them, and your TD and MC code must never use `P` or `R`.

Reinforcement-learning libraries may not be used. NumPy and Matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# ---------------------------------------------------------------
# Personal seed: the last three digits of your roll number.
# Every random experiment in this notebook must use it.
# ---------------------------------------------------------------
SEED = None          # TODO: e.g. SEED = 417

assert SEED is not None, "Set SEED to the last three digits of your roll number before running the notebook."
print("SEED =", SEED)

---
## Provided Starter Code

Run the cells in this section as-is. You do not need to modify them.

| Name | What it gives you |
|---|---|
| `CorridorEnv(p_move=0.8, seed=None)` | `reset()` → `s` ; `step(a)` → `s2, r, done`. The model is hidden, as in Lab 7. |
| `right_policy`, `random_policy` | arrays of shape (5, 2) holding $\pi(a\mid s)$ |
| `run_episode(env, policy, rng)` | → `states, actions, rewards, next_states` (one entry per transition) |
| `run_episode_battery(env, policy, rng, battery)` | the same, but at most `battery` steps; also returns `finished` (Task 6) |
| `V_exact_R`, `V_exact_U` | exact values from Lab 5 (checking only) |
| `plot_values(V, V_ref=None, title)` | bar plot of estimates against a reference |
| `E_states`, `E_rewards`, `E_next_states` | episode E from Lab 7 (Task 2) |

**Note:** `run_episode` now also returns `next_states`, because the TD update needs the cell the robot moved to.

In [ ]:
# ---------------------------------------------------------------
# Warehouse corridor (same dynamics as Lab 5)
#
#   cell:   0     1     2     3     4
#           F     .     S     .     G
#
#   Actions: 0 = Left, 1 = Right
# ---------------------------------------------------------------

N_STATES, N_ACTIONS = 5, 2
ACTIONS = ["Left", "Right"]
TERMINALS = (0, 4)
START = 2
GAMMA = 0.9


def _env_stream_seed(seed, salt):
    """Derives the environment's own seed from `seed`, so that CorridorEnv(seed=k) and
    np.random.default_rng(k) produce INDEPENDENT random streams (otherwise the policy's
    coin flips and the environment's slips would be the same numbers)."""
    if seed is None:
        return None
    return int(np.random.SeedSequence([int(seed), salt]).generate_state(1)[0])


class CorridorEnv:
    """The corridor as an environment you can only ACT in.

    reset()       -> s                 the start cell (2)
    step(action)  -> s2, r, done       next cell, reward, end-of-episode flag

    The slip probability and the arrays P and R are hidden: reading env.P or
    env.R raises an error, exactly as in a real system whose rules you do not know.
    """

    def __init__(self, p_move=0.8, seed=None):
        self.__p_move = p_move                       # hidden dynamics
        self.__rng = np.random.default_rng(_env_stream_seed(seed, 101))
        self.__s = None

    def __getattr__(self, name):
        if name in ("P", "R", "p_move", "transition_matrix"):
            raise AttributeError(f"The model is hidden: '{name}' is not available. "
                                 "This environment only offers reset() and step(action).")
        raise AttributeError(name)

    def reset(self):
        self.__s = START
        return self.__s

    def step(self, action):
        if self.__s is None or self.__s in TERMINALS:
            raise RuntimeError("Episode has ended (or not started): call reset() first.")
        s = self.__s
        if self.__rng.random() < self.__p_move:      # intended move
            s2 = s + (-1 if action == 0 else +1)
        else:                                        # slip: stay in place
            s2 = s
        r = 10.0 if s2 == 4 else (-10.0 if s2 == 0 else -1.0)
        self.__s = s2
        return s2, r, s2 in TERMINALS


# The two policies of Table 3, as pi[s, a] (rows of the terminal cells are never used)
right_policy = np.zeros((N_STATES, N_ACTIONS)); right_policy[:, 1] = 1.0     # pi_R
random_policy = np.full((N_STATES, N_ACTIONS), 0.5)                           # pi_U


def run_episode(env, policy, rng, max_steps=10_000):
    """Plays one complete episode from env.reset() following `policy` (shape (5, 2)).

    Returns four lists with T entries each (one entry per transition):
      states      = [S0, S1, ..., S_{T-1}]
      actions     = [A0, A1, ..., A_{T-1}]
      rewards     = [R1, R2, ..., R_T]          rewards[t] follows action t
      next_states = [S1, S2, ..., S_T]          S_T is the terminal cell (0 or 4)
    """
    s = env.reset()
    states, actions, rewards, next_states = [], [], [], []
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[s]))
        s2, r, done = env.step(a)
        states.append(int(s)); actions.append(a); rewards.append(float(r)); next_states.append(int(s2))
        s = s2
        if done:
            break
    return states, actions, rewards, next_states


def run_episode_battery(env, policy, rng, battery=4):
    """Like run_episode, but the robot's battery allows at most `battery` steps (Task 6).
    The episode may therefore stop BEFORE reaching G or F.

    Returns states, actions, rewards, next_states (as run_episode) and
      finished = True if the episode reached G or F, False if the battery ran out.
    """
    s = env.reset()
    states, actions, rewards, next_states = [], [], [], []
    finished = False
    for _ in range(battery):
        a = int(rng.choice(policy.shape[1], p=policy[s]))
        s2, r, done = env.step(a)
        states.append(int(s)); actions.append(a); rewards.append(float(r)); next_states.append(int(s2))
        s = s2
        if done:
            finished = True
            break
    return states, actions, rewards, next_states, finished

In [ ]:
# ---------------------------------------------------------------
# Exact reference values (from the hidden model, as in Lab 5).
# They are for CHECKING your estimates only: no estimate may be
# computed from them, and your TD and MC code must never use P or R.
# ---------------------------------------------------------------

def _build_corridor(p_move=0.8, step=-1.0, goal=10.0, forklift=-10.0):
    """Hidden Lab 5 model, used only to compute the exact values below."""
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    for t in TERMINALS:
        P[t, :, t] = 1.0
    for s in [1, 2, 3]:
        for a, d in enumerate([-1, +1]):
            s2 = s + d
            r = goal if s2 == 4 else (forklift if s2 == 0 else step)
            P[s, a, s2] += p_move;     R[s, a, s2] = r
            P[s, a, s] += 1 - p_move;  R[s, a, s] = step
    return P, R


def _exact_values(P, R, policy, gamma):
    P_pi = np.einsum("sa,sat->st", policy, P)
    r_pi = np.einsum("sa,sat,sat->s", policy, P, R)
    V = np.linalg.solve(np.eye(P.shape[0]) - gamma * P_pi, r_pi)
    V[list(TERMINALS)] = 0.0
    return V


_P08, _R08 = _build_corridor(0.8)
V_exact_R = _exact_values(_P08, _R08, right_policy, GAMMA) + 0.0    # pi_R, real corridor p = 0.8
V_exact_U = _exact_values(_P08, _R08, random_policy, GAMMA) + 0.0   # pi_U, real corridor p = 0.8
del _P08, _R08

print("V_exact_R =", V_exact_R)
print("V_exact_U =", V_exact_U)


def plot_values(V, V_ref=None, title="", cells=(1, 2, 3)):
    """Bar plot of the estimates V for the non-terminal cells, next to a reference
    (e.g. V_exact_U) if one is given. Cells with no estimate (np.nan) are marked '–'."""
    cells = list(cells)
    x = np.arange(len(cells))
    est = np.array([V[c] for c in cells], dtype=float)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    w = 0.38 if V_ref is not None else 0.6
    ax.bar(x - (w / 2 if V_ref is not None else 0), np.nan_to_num(est), w, label="estimate", color="#4C72B0")
    if V_ref is not None:
        ax.bar(x + w / 2, [V_ref[c] for c in cells], w, label="exact", color="#BBBBBB")
    for i, v in enumerate(est):
        ax.text(x[i] - (w / 2 if V_ref is not None else 0), 0 if np.isnan(v) else v,
                "–" if np.isnan(v) else f"{v:.2f}", ha="center",
                va="bottom" if (np.isnan(v) or v >= 0) else "top", fontsize=9)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([f"cell {c}" for c in cells])
    ax.set_ylabel("V(s)"); ax.set_title(title)
    if V_ref is not None:
        ax.legend()
    plt.tight_layout(); plt.show()


# Episode E from Lab 7 (Task 2): 2 -> 2 -> 3 -> 2 -> 3 -> G
E_states      = [2, 2, 3, 2, 3]
E_actions     = [1, 1, 0, 1, 1]          # 0 = Left, 1 = Right
E_rewards     = [-1, -1, -1, -1, 10]     # E_rewards[t] = R_{t+1}
E_next_states = [2, 3, 2, 3, 4]          # E_next_states[t] = S_{t+1}; cell 4 is G

---
## Part 1: From Monte Carlo to Temporal Difference

Monte Carlo waits until the episode ends and moves the estimate towards the **actual return** $G_t$. Temporal Difference (TD) learning updates the estimate **after every step**, moving it towards a **target built from its own next guess**:

$$\text{MC:}\quad V(S_t) \leftarrow V(S_t) + \alpha\,[\,G_t - V(S_t)\,]$$

$$\text{TD(0):}\quad V(S_t) \leftarrow V(S_t) + \alpha\,[\,\underbrace{R_{t+1} + \gamma V(S_{t+1})}_{\text{TD target}} - V(S_t)\,]$$

The bracket $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the **TD error** ("how surprised am I?") and $\alpha$ is the **learning rate**. The value of a terminal cell is always 0.

### Task 1: What Each Method Needs

**(a)** Table 5: Comparing the three methods

| Question | DP (Lab 5) | Monte Carlo (Lab 7) | TD(0) (Lab 8) |
|---|---|---|---|
| Needs the rules (P and R)? | | | |
| Must wait until the episode ends? | | | |
| Target used in the update | | | |
| Uses its own guesses (bootstraps)? | | | |

**(b)** Complete the analogy. You are driving home and estimating your arrival time. Describe in one sentence each when and how a Monte Carlo driver and a TD driver correct their estimate.

*your answer here*

**Questions**

1. In Lab 7, why could Monte Carlo update nothing until the episode had ended? What does TD use instead of the part it cannot yet see?

   *your answer here*

2. The TD target contains $V(S_{t+1})$, which is itself only a guess. How can learning from a guess ever give the right answer? (Hint: which part of the target is real?)

   *your answer here*

3. Look at the two update rules above. Which single part differs between them?

   *your answer here*

### Task 2: TD Updates by Hand

Use episode E from Lab 7. Start with $V(s) = 0$ for every cell, and use $\alpha = 0.5$ and $\gamma = 0.9$. Apply the TD(0) update once per row, **in order**, always using the most recent values.

**Predict first.** After the first step ($t = 0$), will $V(2)$ go up or down? Why?

*your prediction here*

**(a)** Table 6: TD(0) updates along episode E

| $t$ | $S_t$ | $R_{t+1}$ | $S_{t+1}$ | $V(S_t)$ before | TD target | TD error | $V(S_t)$ after |
|---|---|---|---|---|---|---|---|
| 0 | 2 | –1 | 2 | | | | |
| 1 | 2 | –1 | 3 | | | | |
| 2 | 3 | –1 | 2 | | | | |
| 3 | 2 | –1 | 3 | | | | |
| 4 | 3 | +10 | G | | | | |

*your working here*

**(b)** Table 7: Estimates after episode E alone

| Cell | TD(0) after episode E | First-visit MC from episode E (Lab 7) | Exact value under $\pi_U$ |
|---|---|---|---|
| 2 | | | |
| 3 | | | |

**(c)** Implement `td0_update(V, s, r, s2, alpha, gamma)`, which applies one TD(0) update to the array `V` and returns the TD error. Replay episode E with it and check every row of Table 6.

In [ ]:
def td0_update(V, s, r, s2, alpha, gamma):
    # One TD(0) update of V[s] after the transition s -> s2 with reward r.
    # Terminal cells keep V = 0, so V[s2] can be used directly.
    # Returns the TD error.
    # TODO
    # your code here

    return td_error


# TODO: start from V = 0, replay episode E with td0_update (alpha = 0.5, gamma = GAMMA)
#       and print, for every step, the TD target, the TD error and the new V(S_t).
# your code here

**Questions**

1. After episode E, which estimate (TD or MC) is closer to the exact value in each cell? Why does MC trust this one episode more than TD does?

   *your answer here*

2. The +10 at the end of episode E changed $V(3)$ but not $V(2)$. Why? What would happen to $V(2)$ if the same episode were played a second time?

   *your answer here*

3. Did your prediction about $V(2)$ after step 0 match the table?

   *your answer here*

---
## Part 2: TD(0) Prediction

### Task 3: Implement TD(0) Prediction

Implement `td0_prediction(env, policy, n_episodes, alpha, gamma, rng)`. Unlike Monte Carlo, it must **update while the robot plays**: call `env.step()` yourself and apply `td0_update` after every step. It must return `V` and a record of $V(2)$ after every episode.

In [ ]:
def td0_prediction(env, policy, n_episodes, alpha, gamma, rng):
    # Returns:
    #   V          : array of shape (n_states,), starting from zeros
    #   V2_history : array of shape (n_episodes,), the estimate V(2) after every episode
    # Only env.reset(), env.step() and the policy may be used.
    n_states = policy.shape[0]
    V = np.zeros(n_states)
    V2_history = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s = env.reset()
        done = False
        while not done:
            a = int(rng.choice(policy.shape[1], p=policy[s]))
            s2, r, done = env.step(a)
            # TODO: update V with td0_update, then move to the next cell
            # your code here

        V2_history[ep] = V[2]
    return V, V2_history

**Predict first.** Under $\pi_R$, what do you expect $V(1)$ to be after 5000 episodes, and why?

*your prediction here*

**(a)** Run `td0_prediction` with $\alpha = 0.05$ for $\pi_U$ and $\pi_R$ with 10, 100, 1000 and 5000 episodes (a new `CorridorEnv(seed=SEED)` and `np.random.default_rng(SEED)` for every run), and complete Table 8.

In [ ]:
# TODO: Table 8 — V(2) and its absolute error after 10, 100, 1000 and 5000 episodes
#       for random_policy and right_policy (alpha = 0.05).
# your code here

| Episodes | V(2) under $\pi_U$ | Absolute error | V(2) under $\pi_R$ | Absolute error |
|---|---|---|---|---|
| 10 | | | | |
| 100 | | | | |
| 1000 | | | | |
| 5000 | | | | |

**(b)** For the 5000-episode runs, plot both value functions against the exact values with `plot_values`, and plot $V(2)$ against the number of episodes with a horizontal line at the exact value.

In [ ]:
# TODO: plot_values for both policies, and the V(2) learning curves with the exact value
#       as a horizontal line. Label axes and add titles.
# your code here

**Questions**

1. Under $\pi_R$, what is your estimate of $V(1)$? Is this the same limitation you saw for Monte Carlo in Lab 7? Explain.

   *your answer here*

2. Your Lab 7 Monte Carlo collected returns only after each episode. When exactly did your TD code change $V$? Point to the line of your code.

   *your answer here*

3. After 5000 episodes, the estimate under $\pi_U$ still moves up and down from episode to episode. Use your learning curve to describe this, and suggest what causes it.

   *your answer here*

### Task 4: The Learning Rate α

The learning rate decides how far each update moves the estimate towards the TD target: $\alpha = 1$ jumps all the way, while a small $\alpha$ takes small steps.

**Predict first.** Which of the four values below will reach the exact $V(2)$ fastest, and which will give the steadiest estimate after 1000 episodes?

*your prediction here*

**(a)** For $\pi_U$, run TD(0) for 1000 episodes with $\alpha \in \{0.01, 0.05, 0.1, 0.5\}$ (seed SEED). Plot the four $V(2)$ learning curves on one figure with the exact value as a horizontal line.

In [ ]:
# TODO: four V(2) learning curves (alpha = 0.01, 0.05, 0.1, 0.5) on one labelled figure.
# your code here

**(b)** Repeat each run with ten seeds (SEED, SEED+1, …, SEED+9) and complete Table 9 with the average absolute error of $V(2)$ over the ten runs.

In [ ]:
# TODO: Table 9 — average |V(2) - exact| over ten runs after 10, 100 and 1000 episodes.
# your code here

| α | Average error after 10 episodes | after 100 episodes | after 1000 episodes |
|---|---|---|---|
| 0.01 | | | |
| 0.05 | | | |
| 0.1 | | | |
| 0.5 | | | |

**Questions**

1. Which α learned fastest at the start, and which was most accurate after 1000 episodes? Are they the same α? Quote your numbers.

   *your answer here*

2. Why does a large α keep the estimate jumping around even after many episodes?

   *your answer here*

3. If you had to evaluate a policy with only 50 episodes, which α would you choose? With 5000 episodes? Justify each choice from Table 9.

   *your answer here*

---
## Part 3: TD versus Monte Carlo

### Task 5: A Fair Race

To compare the two methods fairly, give both the **same learning rate** and the **same episodes**, so that the only difference is the target:

$$\text{MC:}\ V(S_t) \leftarrow V(S_t) + \alpha\,[\,G_t - V(S_t)\,] \qquad \text{TD:}\ V(S_t) \leftarrow V(S_t) + \alpha\,[\,R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\,]$$

For MC, update only the **first visit** of each cell in the episode, after the episode has ended, using the returns computed backward as in Lab 7.

**(a)** Implement `mc_alpha_episode(V, states, rewards, alpha, gamma)` and `td0_episode(V, states, rewards, next_states, alpha, gamma)`. Each applies its updates for one recorded episode.

In [ ]:
def returns_backward(rewards, gamma):
    # TODO: paste your Lab 7 returns_backward here
    # your code here

    return G


def mc_alpha_episode(V, states, rewards, alpha, gamma):
    # Constant-alpha first-visit MC: after the episode, move V(S_t) towards G_t
    # for the first visit of each cell.
    # TODO
    # your code here

    pass


def td0_episode(V, states, rewards, next_states, alpha, gamma):
    # TD(0): apply td0_update to every transition of the episode, in order.
    # TODO
    # your code here

    pass

**Predict first.** Which method do you expect to have the smaller error after 50 episodes? After 300?

*your prediction here*

**(b)** For $\pi_U$ with $\alpha = 0.05$, run 20 independent runs of 300 episodes (run $k$ uses seed SEED + $k$). In every run, generate each episode once with `run_episode` and apply **both** updates to it (each method has its own V, starting from zeros). After every episode, record the root-mean-square (RMS) error over cells 1, 2 and 3:

$$\text{RMS error} = \sqrt{\tfrac{1}{3}\sum_{s=1}^{3}\big(V(s) - V_{\text{exact}}(s)\big)^2}$$

Plot the RMS error averaged over the 20 runs against the number of episodes, one line per method, and complete Table 10.

In [ ]:
# TODO: 20 runs x 300 episodes, the same episodes for both methods, average RMS error
#       curves for TD and MC on one labelled plot, and Table 10.
# your code here

| Episodes | Average RMS error, MC | Average RMS error, TD(0) | Smaller error |
|---|---|---|---|
| 10 | | | |
| 50 | | | |
| 100 | | | |
| 300 | | | |

**Questions**

1. Which method had the smaller error in your race, and at which numbers of episodes? Quote Table 10.

   *your answer here*

2. The MC target $G_t$ depends on every random slip and coin flip until the end of the episode, while the TD target depends on only one step. Use this to explain your result.

   *your answer here*

3. In Lab 7, MC averaged all its returns instead of using a fixed α. Why do we give both methods the same α here?

   *your answer here*

4. Did both curves stop improving at some level? Relate this to what you found about a fixed α in Task 4.

   *your answer here*

---
## Part 4: Learning from Unfinished Episodes

### Task 6: When the Battery Runs Out

The robot's battery now allows at most **4 steps** per episode. Many episodes therefore stop before the robot reaches G or F. `run_episode_battery(env, policy, rng, battery=4)` plays such an episode and also returns `finished`.

- **Monte Carlo** needs the return $G_t$, which is only known if the episode reached G or F, so it can use **only finished episodes**.
- **TD** can update after every step it has seen. When an episode stops early, the last cell is **not** terminal, so its TD target still uses $\gamma V(S_{t+1})$.

**Predict first.** About what fraction of the episodes do you think will finish within 4 steps under $\pi_U$? Which method will be more accurate?

*your prediction here*

**(a)** For $\pi_U$ with $\alpha = 0.05$, play 1000 battery-limited episodes (seed SEED). Update TD on every episode and MC (constant α, first visit) on finished episodes only. Record the number of finished episodes and the number of updates each method made, and complete Table 11.

In [ ]:
# TODO: 1000 battery-limited episodes; TD on all of them, MC on finished ones only.
#       Count finished episodes and updates; compare both estimates with V_exact_U.
# your code here

| Measure | Monte Carlo | TD(0) |
|---|---|---|
| Episodes used | | |
| Number of updates made | | |
| Estimate of V(1) | | |
| Estimate of V(2) | | |
| Estimate of V(3) | | |
| RMS error over cells 1–3 | | |

Episodes that finished within 4 steps: ___ out of 1000

**(b)** Plot both estimates against the exact values with `plot_values`.

In [ ]:
# TODO: plot_values for the MC and TD estimates of Task 6.
# your code here

**Questions**

1. How many episodes did Monte Carlo have to throw away? What fraction of the robot's experience was lost?

   *your answer here*

2. The MC estimate of V(3) is far too high. The finished episodes are not a fair sample of all episodes: which kind of episode finishes within 4 steps from cell 3? Explain why this biases the estimate.

   *your answer here*

3. Why can TD still learn correctly from an episode that stopped at, say, cell 1? What does it use in place of the missing future?

   *your answer here*

4. Name one real task where episodes are very long or never end. Which method would you use, and why?

   *your answer here*

---
## Part 5: Summary and Reflection

### Task 7: Summary and Reflection

**(a)** Table 12: Dynamic programming, Monte Carlo and TD compared

| Property | DP (Lab 5) | Monte Carlo (Lab 7) | TD(0) (Lab 8) |
|---|---|---|---|
| Needs the rules (P and R) | | | |
| When the estimate is updated | | | |
| Target of the update | | | |
| Can use unfinished episodes | | | |
| Main source of error in your results | | | |

**(b)** In no more than 150 words, explain what TD learning offers compared with Monte Carlo, and name one weakness. Quote at least two numbers from your own results. State what would have to change to move from estimating a policy to improving it with TD.

*your answer here (max 150 words)*